In [45]:
#unzip all files

import os
import zipfile

def unzip_all(zip_dir, extract_dir):
    for zip_file in os.listdir(zip_dir):
        if zip_file.endswith(".zip"):
            zip_path = os.path.join(zip_dir, zip_file)
            folder_name = os.path.splitext(zip_file)[0]  # Remove .zip extension
            extract_path = os.path.join(extract_dir, folder_name)
            
            os.makedirs(extract_path, exist_ok=True)  # Create a directory for extraction
            
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_path)
            
            print(f"Extracted {zip_file} to {extract_path}")

zip_directory = "/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2"
output_directory = zip_directory

unzip_all(zip_directory, output_directory)


Extracted 9GNJ_PS.result.zip to /media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/9GNJ_PS.result
Extracted 7X6Z_PS.result.zip to /media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/7X6Z_PS.result
Extracted 8D7N_PS.result.zip to /media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/8D7N_PS.result
Extracted 8DKV_PS.result.zip to /media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/8DKV_PS.result
Extracted 8DKN_PS.result.zip to /media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/8DKN_PS.result
Extracted 8FSR_PS.result.zip to /media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/8FSR_PS.result
Extracted 7TB1_PS.result.zip to /media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/7TB1_PS.result
Extracted 8Q1N_PS.result.zip to /media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/8Q1N_PS.result
Extracted 8UO7_PS.result.zip to /media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/8UO7_PS.result
Extracted 7YKF_PS.result.zip to /media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/7YKF_PS.result
Extracted 9KD5_PS.result.zip to /media/emel/d/slim_af2.3/NEW/AF2_v23_p

In [2]:
import pandas as pd
import os

pred_method = "af2"

original_directory = "/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/original_pdbs" ## Native PDB directory
folder_path = f"/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/{pred_method}"
complex_list2 = [
    f for f in os.listdir(folder_path)
    if os.path.isdir(os.path.join(folder_path, f)) and not f.startswith(".")
]
# print(complex_list2)  # should print prediction folders ['7ar0_B_A_af2', '...', '...']

complex_list = [f.replace(".result","") for f in complex_list2]

# print(complex_list)
## file hierarchy should be like this
# af2
# ├── 7ar0_B_A_af2
# │   ├── 7ar0_B_A_af2.a3m
# │   ├── 7ar0_B_A_af2.png
# │   ├── 7ar0_B_A_af2.png
# │   ├── 7ar0_B_A_af2.png
# │   ├── ...
# ├── 7bnv_H_L_A_af2
# ├── ...

## create results folder
result_folder = pred_method + "_results"
result_path = os.path.join("/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/", result_folder)
os.makedirs(result_path, exist_ok=True)
dockq_output = f"{pred_method}_dockq_fnat_scores.csv"  ## dockq output
pdockq_output = f"{pred_method}_pdockq2_fit.csv" ## pdockq2 output
combined = f"{pred_method}_combined.csv" ### combined results with dockq_pdock2 and model scores

# print(result_path)

In [3]:
import os
import sys
import csv
import glob
import pandas as pd
from statistics import mean

# Ensure DockQ is in your path or installed
try:
    from DockQ.DockQ import load_PDB, run_on_all_native_interfaces
except ImportError:
    print("Error: Could not import DockQ. Please ensure the DockQ repository is in your PYTHONPATH.")
    sys.exit(1)

# --- CONFIGURATION ---
# Replace these with your actual paths
ROOT_INPUT_FOLDER = folder_path 
ORIGINAL_PDB_FOLDER = original_directory

FINAL_SUMMARY_FILE = "all_dockq_scores.csv"
ERROR_LOG_FILE = "error_log.txt"
# ---------------------

def log_error(message, error_file_path):
    """Appends an error message to the log file."""
    print(f"XXX ERROR: {message}")
    with open(error_file_path, "a") as f:
        f.write(f"{message}\n")

def merge_chains(model, chains_to_merge):
    """Merges specified chains in the given model."""
    for chain in chains_to_merge[1:]:
        for res in list(model[chain]):
            res.id = (chains_to_merge[0], res.id[1], res.id[2])
            model[chains_to_merge[0]].add(res)
        model.detach_child(chain)
    model[chains_to_merge[0]].id = "".join(chains_to_merge)
    return model

def calculate_dockq(model, native, chain_map):
    """Calculates DockQ scores."""
    try:
        results, dockq_score = run_on_all_native_interfaces(model, native, chain_map=chain_map)
        return results, dockq_score
    except Exception as e:
        raise RuntimeError(f"DockQ internal calculation error: {e}")

def process_models(models, error_log_path):
    """Processes the provided models and calculates DockQ scores."""
    results_list = []
    
    for model_file, native_file in models:
        # --- CHANGE: Use exact filename for model_id ---
        model_id = os.path.basename(model_file)
        # -----------------------------------------------
        native_id = os.path.basename(native_file)
        
        print(f"Processing model: {model_id}")
        
        try:
            model = load_PDB(model_file)
            native = load_PDB(native_file)
        except Exception as e:
            log_error(f"{model_id}: Failed to load PDB files. Error: {e}", error_log_path)
            continue

        chain_ids = list(model.child_dict.keys())
        native_chain_ids = list(native.child_dict.keys())

        try:
            # --- Logic for 3 chains (Merge first two) ---
            if len(chain_ids) == 3:
                # Validate Native Chains exist before proceeding
                if len(native_chain_ids) < 3:
                     # Flag mismatch if native lacks required chains
                     raise ValueError(f"Model has 3 chains {chain_ids} but Native {native_id} only has {native_chain_ids}")

                # Proceed with merge
                model_merged = merge_chains(model, chain_ids[:2])
                native_merged = merge_chains(native, native_chain_ids[:2])
                
                # Check merged result keys
                curr_model_chains = list(model_merged.child_dict.keys())
                curr_native_chains = list(native_merged.child_dict.keys())
                
                # Ensure we have the chains we expect after merge
                if len(curr_native_chains) < 2:
                    raise ValueError(f"Native structure chain merge failed. Resulting chains: {curr_native_chains}")

                chain_map_merged = {curr_native_chains[1]: curr_model_chains[1], curr_native_chains[0]: curr_model_chains[0]}
                
                results_merged, _ = calculate_dockq(model_merged, native_merged, chain_map_merged)
                
                if results_merged:
                    merged_result = results_merged[list(results_merged.keys())[0]]
                    results_list.append((
                        model_id, merged_result['DockQ'], merged_result['fnat'],
                        merged_result['iRMSD'], merged_result['LRMSD'], merged_result['F1']
                    ))

            # --- Logic for 2 chains ---
            elif len(chain_ids) == 2:
                # Validate Native Chains
                if len(native_chain_ids) < 2:
                     raise ValueError(f"Model has 2 chains {chain_ids} but Native {native_id} only has {native_chain_ids}")
                
                # Safer Mapping: verify the native actually has the chains corresponding to index 0 and 1
                try:
                    target_native_chain_1 = native_chain_ids[0]
                    target_native_chain_2 = native_chain_ids[1]
                except IndexError:
                    raise ValueError(f"Native {native_id} missing chains at index 0 or 1.")

                chain_map = {target_native_chain_1: chain_ids[0], target_native_chain_2: chain_ids[1]}
                
                results, _ = calculate_dockq(model, native, chain_map)
                
                if results:
                    first_key = list(results.keys())[0]
                    results_list.append((
                        model_id, results[first_key]['DockQ'], results[first_key]['fnat'],
                        results[first_key]['iRMSD'], results[first_key]['LRMSD'], results[first_key]['F1']
                    ))
            
            else:
                log_error(f"{model_id}: Skipping. Model has {len(chain_ids)} chains (only 2 or 3 supported).", error_log_path)

        except KeyError as e:
            log_error(f"{model_id}: Chain ID Error. Missing chain in native or model. Details: {e}", error_log_path)
        except ValueError as e:
            log_error(f"{model_id}: Structure Mismatch. {e}", error_log_path)
        except Exception as e:
            log_error(f"{model_id}: Unexpected Error during processing. {e}", error_log_path)

    return results_list

def save_results_to_csv(results, filename):
    if not results:
        return
    print(f"Saving results to: {filename}")
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['model_id', 'DockQ', 'fnat', "iRMSD", "LRMSD", "F1"])
        for row in results:
            writer.writerow(row)

def process_folder(current_dir, original_directory, error_log_path):
    # 1. Find Model Files
    pdb_files = glob.glob(os.path.join(current_dir, "*_unrelaxed_rank_0*.pdb"))
    if not pdb_files:
        return None

    models = []
    base_name_for_csv = ""

    # 2. Match with Natives
    for pdb_file in pdb_files:
        pdb_filename = os.path.basename(pdb_file)
        # Parse ID: "1A2B_unrelaxed..." -> "1A2B"
        pdb_id_base = pdb_filename.split('_unrelaxed')[0] 
        
        # NOTE: Adjust this logic if your naming is complex (e.g. 1A2B_A_B)
        # pdb_id_base = "_".join(pdb_id_base.split("_")[:-1]) 

        native_candidate = os.path.join(original_directory, f"{pdb_id_base}.pdb")
        
        if os.path.exists(native_candidate):
            models.append((pdb_file, native_candidate))
            base_name_for_csv = pdb_id_base
        else:
            log_error(f"{pdb_filename}: Native file not found at {native_candidate}", error_log_path)

    if not models:
        return None

    # 3. Run Calculations
    results = process_models(models, error_log_path)

    # 4. Save Local CSV
    if results:
        csv_name = f"{base_name_for_csv}_dockq_scores.csv" if base_name_for_csv else "dockq_scores.csv"
        output_path = os.path.join(current_dir, csv_name)
        save_results_to_csv(results, output_path)
        return output_path
    
    return None

def main():
    print(f"Starting Scan in: {ROOT_INPUT_FOLDER}")
    print(f"Using Natives from: {ORIGINAL_PDB_FOLDER}")
    
    error_log_path = os.path.join(ROOT_INPUT_FOLDER, ERROR_LOG_FILE)
    # Clear previous error log
    with open(error_log_path, "w") as f:
        f.write("--- DockQ Processing Error Log ---\n")

    all_csv_files = []

    # Walk through all subdirectories
    for root, dirs, files in os.walk(ROOT_INPUT_FOLDER):
        if any("_unrelaxed_rank_0" in f for f in files):
            print(f"\n--- Processing Folder: {root} ---")
            created_csv = process_folder(root, ORIGINAL_PDB_FOLDER, error_log_path)
            if created_csv:
                all_csv_files.append(created_csv)

    # Combine all results
    print("\n--- Combining Results ---")
    if all_csv_files:
        df_list = []
        for file in all_csv_files:
            try:
                df = pd.read_csv(file)
                df['Source_Folder'] = os.path.dirname(file)
                df_list.append(df)
            except pd.errors.EmptyDataError:
                pass
        
        if df_list:
            combined_df = pd.concat(df_list, ignore_index=True)
            output_file = os.path.join(ROOT_INPUT_FOLDER, FINAL_SUMMARY_FILE)
            combined_df.to_csv(output_file, index=False)
            print(f"Success! Combined scores saved to: {output_file}")
        else:
            print("No valid data found in CSVs.")
    else:
        print("No results generated.")
        
    print(f"\nProcessing Complete. Check {error_log_path} for any failed files.")

if __name__ == "__main__":
    main()

Starting Scan in: /media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2
Using Natives from: /media/emel/d/slim_af2.3/NEW/AF2_v23_pred/original_pdbs

--- Processing Folder: /media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/9OLB_PS.result ---
Processing model: 9OLB_PS_unrelaxed_rank_011_alphafold2_multimer_v3_model_4_seed_002.r3.pdb
Processing model: 9OLB_PS_unrelaxed_rank_001_alphafold2_multimer_v3_model_2_seed_004.pdb
Processing model: 9OLB_PS_unrelaxed_rank_004_alphafold2_multimer_v3_model_3_seed_001.r0.pdb
Processing model: 9OLB_PS_unrelaxed_rank_002_alphafold2_multimer_v3_model_3_seed_000.r0.pdb
Processing model: 9OLB_PS_unrelaxed_rank_014_alphafold2_multimer_v3_model_5_seed_001.r1.pdb
Processing model: 9OLB_PS_unrelaxed_rank_007_alphafold2_multimer_v3_model_1_seed_003.r1.pdb
Processing model: 9OLB_PS_unrelaxed_rank_015_alphafold2_multimer_v3_model_4_seed_000.r2.pdb
Processing model: 9OLB_PS_unrelaxed_rank_011_alphafold2_multimer_v3_model_4_seed_002.pdb
Processing model: 9OLB_PS_unrelaxed_rank_

In [5]:
import pandas as pd

def classify_dockq(score):
    if 0.00 <= score < 0.23:
        return 'Incorrect'
    elif 0.23 <= score < 0.49:
        return 'Acceptable'
    elif 0.49 <= score < 0.80:
        return 'Medium'
    elif score >= 0.80:
        return 'High'
    else:
        return 'Invalid score'

def classify_capri(row):
    fnat, i_rmsd, l_rmsd = row['fnat'], row['iRMSD'], row['LRMSD']

    # High: fnat ≥ 0.5 AND (L-RMSD ≤ 1.0 OR i-RMSD ≤ 1.0)
    if fnat >= 0.5 and (l_rmsd <= 1.0 or i_rmsd <= 1.0):
        return "High"

    # Medium: fnat ≥ 0.3 AND (L-RMSD ≤ 5.0 OR i-RMSD ≤ 2.0)
    if fnat >= 0.3 and (l_rmsd <= 5.0 or i_rmsd <= 2.0):
        return "Medium"

    # Acceptable: fnat ≥ 0.1 AND (L-RMSD ≤ 10.0 OR i_RMSD ≤ 4.0)
    if fnat >= 0.1 and (l_rmsd <= 10.0 or i_rmsd <= 4.0):
        return "Acceptable"

    # Otherwise, Incorrect
    return "Incorrect"

def classify_capri_peptide(row):
    fnat, i_rmsd, l_rmsd = row['fnat'], row['iRMSD'], row['LRMSD']

    # High: fnat ∈ [0.8, 1.0] AND (L-RMSD ≤ 1.0 OR i-RMSD ≤ 0.5)
    if 0.8 <= fnat <= 1.0 and (l_rmsd <= 1.0 or i_rmsd <= 0.5):
        return "High"

    # Medium:
    # Case 1: fnat ∈ [0.5, 0.8] AND (L-RMSD ≤ 2.0 OR i-RMSD ≤ 1.0)
    # OR Case 2: fnat ∈ [0.8, 1.0] AND (L-RMSD > 1.0 AND i-RMSD > 0.5)
    if (0.5 <= fnat < 0.8 and (l_rmsd <= 2.0 or i_rmsd <= 1.0)) or \
       (0.8 <= fnat <= 1.0 and (l_rmsd > 1.0 and i_rmsd > 0.5)):
        return "Medium"

    # Acceptable:
    # Case 1: fnat ∈ [0.2, 0.5] AND (L-RMSD ≤ 4.0 OR i-RMSD ≤ 2.0)
    # OR Case 2: fnat ∈ [0.5, 1.0] AND (L-RMSD > 2.0 AND i-RMSD > 1.0)
    if (0.2 <= fnat < 0.5 and (l_rmsd <= 4.0 or i_rmsd <= 2.0)) or \
       (0.5 <= fnat <= 1.0 and (l_rmsd > 2.0 and i_rmsd > 1.0)):
        return "Acceptable"

    # All else: Incorrect
    return "Incorrect"
df=pd.read_csv("/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/all_dockq_scores.csv")
print(list(df))
df['CAPRI'] = df.apply(classify_capri, axis=1)
df['CAPRIp'] = df.apply(classify_capri_peptide, axis=1)
df['DockQ_category'] = df['DockQ'].apply(classify_dockq)
# df["model_confidence"] = 0.8*df["ipTM"]+0.2 *df["pTM"]

print(list(df))
df.to_csv(f"{result_path}/{dockq_output}")
df

['model_id', 'DockQ', 'fnat', 'iRMSD', 'LRMSD', 'F1', 'Source_Folder']
['model_id', 'DockQ', 'fnat', 'iRMSD', 'LRMSD', 'F1', 'Source_Folder', 'CAPRI', 'CAPRIp', 'DockQ_category']


,model_id,DockQ,fnat,iRMSD,LRMSD,F1,Source_Folder,CAPRI,CAPRIp,DockQ_category
0,9OLB_PS_unrelaxed_rank_011_alphafold2_multimer...,0.220548,0.083333,3.553629,9.844833,0.075949,/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/...,Incorrect,Incorrect,Incorrect
1,9OLB_PS_unrelaxed_rank_001_alphafold2_multimer...,0.206920,0.027778,3.481098,9.659983,0.025641,/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/...,Incorrect,Incorrect,Incorrect
2,9OLB_PS_unrelaxed_rank_004_alphafold2_multimer...,0.196738,0.055556,3.739000,10.496975,0.057971,/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/...,Incorrect,Incorrect,Incorrect
3,9OLB_PS_unrelaxed_rank_002_alphafold2_multimer...,0.176925,0.027778,4.002714,10.860631,0.027397,/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/...,Incorrect,Incorrect,Incorrect
4,9OLB_PS_unrelaxed_rank_014_alphafold2_multimer...,0.204053,0.083333,3.865840,10.454474,0.077922,/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/...,Incorrect,Incorrect,Incorrect
...,...,...,...,...,...,...,...,...,...,...
35024,7YXP_PS_unrelaxed_rank_002_alphafold2_multimer...,0.792168,0.944444,1.371400,3.028374,0.894737,/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/...,Medium,Medium,Medium
35025,7YXP_PS_unrelaxed_rank_012_alphafold2_multimer...,0.810866,0.944444,1.270441,2.740166,0.894737,/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/...,Medium,Medium,High
35026,7YXP_PS_unrelaxed_rank_018_alphafold2_multimer...,0.818366,0.944444,1.228575,2.637849,0.906667,/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/...,Medium,Medium,High
35027,7YXP_PS_unrelaxed_rank_011_alphafold2_multimer...,0.792204,0.944444,1.368327,3.043576,0.894737,/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/...,Medium,Medium,Medium


In [6]:
### pdockq2 functions

from Bio.PDB import PDBIO
from Bio.PDB.PDBParser import PDBParser
from Bio.PDB.Selection import unfold_entities
import numpy as np
import sys,os
import argparse
import pickle
import itertools
import pandas as pd
from scipy.optimize import curve_fit

def retrieve_IFplddt(structure, chain1, chain2_lst, max_dist):
    chain_lst = list(chain1) + chain2_lst
    ifplddt = []
    contact_chain_lst = []
    for res1 in structure[0][chain1]:
        for chain2 in chain2_lst:
            count = 0
            for res2 in structure[0][chain2]:
                if res1.has_id('CA') and res2.has_id('CA'):
                   dis = abs(res1['CA']-res2['CA'])
                   ## add criteria to filter out disorder res
                   if dis <= max_dist:
                      ifplddt.append(res1['CA'].get_bfactor())
                      count += 1
                elif res1.has_id('CB') and res2.has_id('CB'):
                   dis = abs(res1['CB']-res2['CB'])
                   if dis <= max_dist:
                      ifplddt.append(res1['CB'].get_bfactor())
                      count += 1
            if count > 0:
              contact_chain_lst.append(chain2)
    contact_chain_lst = sorted(list(set(contact_chain_lst)))   
    if len(ifplddt)>0:
       IF_plddt_avg = np.mean(ifplddt)
    else:
       IF_plddt_avg = 0
    return IF_plddt_avg, contact_chain_lst


def retrieve_IFPAEinter(structure, paeMat, contact_lst, max_dist):
    chain_lst = [x.id for x in structure[0]]
    seqlen = [len(x) for x in structure[0]]
    ifch1_col=[]
    ifch2_col=[]
    ch1_lst=[]
    ch2_lst=[]
    ifpae_avg = []
    d=10
    for ch1_idx in range(len(chain_lst)):
      idx = chain_lst.index(chain_lst[ch1_idx])
      ch1_sta=sum(seqlen[:idx])
      ch1_end=ch1_sta+seqlen[idx]
      ifpae_col = []   
      for contact_ch in contact_lst[ch1_idx]:
        index = chain_lst.index(contact_ch)
        ch_sta = sum(seqlen[:index])
        ch_end = ch_sta+seqlen[index]
        paeMat = np.array(paeMat)
        remain_paeMatrix = paeMat[ch1_sta:ch1_end,ch_sta:ch_end]
        mat_x = -1
        for res1 in structure[0][chain_lst[ch1_idx]]:
          mat_x += 1
          mat_y = -1
          for res2 in structure[0][contact_ch]:
              mat_y+=1
              if res1['CA'] - res2['CA'] <=max_dist:
                 ifpae_col.append(remain_paeMatrix[mat_x,mat_y])
      if not ifpae_col:
        ifpae_avg.append(0)
      else:
        norm_if_interpae=np.mean(1/(1+(np.array(ifpae_col)/d)**2))
        ifpae_avg.append(norm_if_interpae)
    return ifpae_avg

def calc_pmidockq(ifpae_norm, ifplddt):
    df = pd.DataFrame()
    df['ifpae_norm'] = ifpae_norm
    df['ifplddt'] = ifplddt
    df['prot'] = df.ifpae_norm*df.ifplddt
    fitpopt = [1.31034849e+00, 8.47326239e+01, 7.47157696e-02, 5.01886443e-03] ## from orignal fit function  
    df['pmidockq'] = sigmoid(df.prot.values, *fitpopt)
    return df

def sigmoid(x, L ,x0, k, b):
    y = L / (1 + np.exp(-k*(x-x0)))+b
    return (y)

def process_pdb_file(pdb_file, json_file, distance, file_id, chains_part=""): 
    pdbp = PDBParser(QUIET=True)
    structure = pdbp.get_structure('', pdb_file)
    chains = [chain.id for chain in structure[0]]
    remain_contact_lst = []
    plddt_lst = []
    for idx in range(len(chains)):
        chain2_lst = list(set(chains)-set(chains[idx]))
        IF_plddt, contact_lst = retrieve_IFplddt(structure, chains[idx], chain2_lst, distance)
        plddt_lst.append(IF_plddt)
        remain_contact_lst.append(contact_lst)
    pae_data = pd.read_json(json_file, lines=True)
    avgif_pae = retrieve_IFPAEinter(structure, pae_data["pae"][0], remain_contact_lst, distance)
    res = calc_pmidockq(avgif_pae, plddt_lst)
    pdb_id = os.path.basename(pdb_file).split('_')[0]
    result = {
        "model_id" : pdb_file,
        "pdb_id": file_id,
        "pdb_id_with_chains": '{0}_{1}'.format(pdb_id, chains_part),
        "ipae_norm_ag": res['ifpae_norm'].tolist()[-1],
        "ipae_norm_avg": np.mean(res['ifpae_norm']), 
        "iplddt_ag": res['ifplddt'].tolist()[-1],
        "iplddt_avg": np.mean(res['ifplddt']), 
        "pDockQ2_ag": res['pmidockq'].tolist()[-1],
        "pDockQ2_avg": np.mean(res['pmidockq'])}
    return result

In [ ]:
import os
import glob
import pandas as pd

# --- Mocking missing variables (Ensure you have these defined in your real script) ---
# result_path = "/path/to/results"
# folder_path = "/path/to/data"
# complex_list = ["9OLB_PS", "ANOTHER_ID"] 
# pdockq_output = "combined_results.csv"
# -----------------------------------------------------------------------------------

def find_matching_json(pdb_file, directory):
    pdb_file_basename = os.path.basename(pdb_file)
    parts = pdb_file_basename.split('_')
    pdb_id = parts[0]
    
    # robust parsing for "unrelaxed"
    try:
        # Find the index of the part starting with 'unrelaxed'
        unrelaxed_idx = next(i for i, p in enumerate(parts) if p.startswith('unrelaxed'))
        chains_part = "_".join(parts[1:unrelaxed_idx])
    except StopIteration:
        print(f"Skipping {pdb_file_basename}: 'unrelaxed' keyword not found in filename.")
        return None

    pattern = f"{pdb_id}_{chains_part}_scores_rank_0*.json"
    json_files = glob.glob(os.path.join(directory, pattern))
    
    if json_files:
        return json_files[0]
    else:
        print(f"Pattern {pattern} not found in {directory}")
        return None

def run_processing(directory, result_output_path):
    # Search for PDB files in the specific directory
    pdb_files = glob.glob(os.path.join(directory, "*_unrelaxed_rank_0*.pdb"))
    
    results = []    
    current_file_id = "unknown_complex" 

    for pdb_file in pdb_files:
        pdb_file_basename = os.path.basename(pdb_file)
        parts = pdb_file_basename.split('_')
        pdb_id = parts[0]
        
        # Re-calculate chains_part safely
        try:
            unrelaxed_idx = next(i for i, p in enumerate(parts) if p.startswith('unrelaxed'))
            chains_part = "_".join(parts[1:unrelaxed_idx])
        except StopIteration:
            continue

        json_file_name = find_matching_json(pdb_file, directory)
        
        # Check if json_file_name is valid (not None) AND exists
        if json_file_name and os.path.exists(json_file_name):
            print(f"Processing: {pdb_id}_{chains_part}")
            
            # Update file_id for naming the CSV later
            current_file_id = f"{pdb_id}_{chains_part}"
            
            try:
                # Ensure process_pdb_file is defined in your actual script
                result = process_pdb_file(pdb_file, json_file_name, 8, pdb_id, chains_part) 
                print(f"Result for {pdb_file}: {result}")  
                results.append(result)
            except NameError:
                print("Error: 'process_pdb_file' function is not defined.")
                return
        else:
            print(f"JSON file for {pdb_file} not found or invalid.")
            continue # Changed from return to continue to allow other PDBs in folder to process

    if results:
        df = pd.DataFrame(results)
        
        csv_filename = f"{current_file_id}_pdockq2_fit.csv"
        csv_file_path = os.path.join(result_output_path, csv_filename)
        
        # Ensure output directory exists
        os.makedirs(result_output_path, exist_ok=True)
        
        df.to_csv(csv_file_path, index=False)
        print(f"Data saved to {csv_file_path}")
    else:
        print(f"No results generated for {directory}")

def combine_csv_files(result_path, output_file=None):
    csv_files = glob.glob(os.path.join(result_path, "*_pdockq2_fit.csv"))
    if not csv_files:
        print("No CSV files found to combine.")
        return pd.DataFrame()

    df_list = [pd.read_csv(file) for file in csv_files]
    combined_df = pd.concat(df_list, ignore_index=True)
    
    if output_file:
        output_path = os.path.join(result_path, output_file)
        combined_df.to_csv(output_path, index=False)
        print(f"Combined CSV saved to {output_path}")
    
    return combined_df

# --- Main Execution Loop (UPDATED) ---

# Ensure result_path is absolute to avoid relative path errors
# result_path = os.path.abspath(result_path) 

print(f"Starting batch processing on {len(complex_list)} items...")

for i in complex_list:
    # 1. FIX: Check if the ID needs the .result suffix appended
    folder_name = i if i.endswith('.result') else f"{i}.result"
    
    # 2. Construct the full path
    fpath = os.path.join(folder_path, folder_name)
    
    # 3. FIX: Check if the folder actually exists before trying to process
    if os.path.exists(fpath):
        print(f"--- Analyzing folder: {fpath} ---")
        run_processing(fpath, result_path)
    else:
        print(f"❌ Skipping: Folder not found at {fpath}")

# Optional: Run the combiner at the end
combined_df = combine_csv_files(result_path, pdockq_output)

Starting batch processing on 348 items...
--- Analyzing folder: /media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/9OLB_PS.result ---
Processing: 9OLB_PS
Result for /media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/9OLB_PS.result/9OLB_PS_unrelaxed_rank_011_alphafold2_multimer_v3_model_4_seed_002.r3.pdb: {'model_id': '/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/9OLB_PS.result/9OLB_PS_unrelaxed_rank_011_alphafold2_multimer_v3_model_4_seed_002.r3.pdb', 'pdb_id': '9OLB', 'pdb_id_with_chains': '9OLB_PS', 'ipae_norm_ag': 0.444254536168753, 'ipae_norm_avg': 0.5411987185174538, 'iplddt_ag': 36.35257142857142, 'iplddt_avg': 56.87342857142857, 'pDockQ2_ag': 0.012770441499723601, 'pDockQ2_avg': 0.05250378299323086}
Processing: 9OLB_PS
Result for /media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/9OLB_PS.result/9OLB_PS_unrelaxed_rank_001_alphafold2_multimer_v3_model_2_seed_004.pdb: {'model_id': '/media/emel/d/slim_af2.3/NEW/AF2_v23_pred/af2/9OLB_PS.result/9OLB_PS_unrelaxed_rank_001_alphafold2_multimer_v3_mode

In [ ]:
import pandas as pd
import re

def parse_log_file(log_file):
    data = []
    query_pdb = "Unknown" 

    with open(log_file, 'r') as f:
        for line in f:
            line = line.strip()
            
            # --- 1. Capture Query Name ---
            match_query = re.search(r'Query \d+/\d+: (\S+) ', line)
            if match_query:
                query_pdb = match_query.group(1)
                continue # Move to next line

            # --- 2. Check for Recycle Lines (contains "recycle=") ---
            # Format: [Name] recycle=[N] pLDDT=[...] ...
            match_recycle = re.search(r'(\S+seed_\d+)\s+recycle=(\d+)\s+pLDDT=([\d.]+)\s+pTM=([\d.]+)\s+ipTM=([\d.]+)', line)
            
            if match_recycle:
                base_name = match_recycle.group(1)
                recycle_num = match_recycle.group(2)
                
                # Convert scores to floats
                plddt = float(match_recycle.group(3))
                ptm = float(match_recycle.group(4))
                iptm = float(match_recycle.group(5))
                
                # Format Name: Base + .rN + .pdb
                final_name = f"{base_name}.r{recycle_num}.pdb"
                
                # Recycle lines usually don't have actifpTM, so we pass None
                data.append([query_pdb, final_name, plddt, ptm, iptm, None])
                continue

            # --- 3. Check for Standard/Rank Lines (NO "recycle=") ---
            # Format: [rank_...Name] pLDDT=[...] ...
            # We look for "pLDDT=" but ensure "recycle=" is NOT in the line (implicit by order or specific regex)
            match_rank = re.search(r'(\S+seed_\d+)\s+pLDDT=([\d.]+)\s+pTM=([\d.]+)\s+ipTM=([\d.]+)(?:\s+actifpTM=([\d.]+))?', line)
            
            if match_rank:
                base_name = match_rank.group(1)
                
                # Convert scores
                plddt = float(match_rank.group(2))
                ptm = float(match_rank.group(3))
                iptm = float(match_rank.group(4))
                
                # Check if actifpTM exists (it's optional in the regex)
                actifptm = float(match_rank.group(5)) if match_rank.group(5) else None

                # Format Name: Base + .pdb
                final_name = f"{base_name}.pdb"
                
                data.append([query_pdb, final_name, plddt, ptm, iptm, actifptm])
                continue

    # Create DataFrame
    df = pd.DataFrame(data, columns=["Query_PDB", "Model_Name", "pLDDT", "pTM", "ipTM", "actifpTM"])
    return df

# --- How to run it ---
dfy = parse_log_file(f"{folder_path}/log.txt")
dfy
# print(df.head())
# dfy['Model_Name']=dfy['Model_Name'].str.replace("rank","unrelaxed_rank")
dfy["model_confidence"] = 0.8*dfy["ipTM"]+0.2 *dfy["pTM"]
dfy.to_csv(f"{result_path}/extracted_metrics.csv", index=False)